# UPCT Medical Image Segmentation Challenge 2026-27
## Práctica 10

**Asignatura:** Procesado de Imágenes Médicas (521104007)

**Profesor:** Juan Zapata

> Contenido **nuevo** de esta práctica. Copia las celdas de aquí abajo y pégalas **al final** de tu propio notebook (el que empezaste en la Práctica 6) — no repitas las prácticas anteriores, ya las tienes hechas ahí.
>
> **Entrega final:** una vez pegadas estas celdas, tu notebook contendrá el proyecto completo (Prácticas 6-10). Renómbralo como `PIM_Challenge_Student_apellido1_apellido2_nombre.ipynb` (sustituye por tus apellidos y nombre reales) y súbelo al aula virtual junto con tu `submission.csv` (o `submission_TTA.csv`, el mejor que hayas subido) — el mismo fichero que ya subiste al Leaderboard de Kaggle.

## Guía de Sesiones (2 horas por sesión)
| Práctica | Fechas (Grupo A / B) | Objetivo de la Sesión | Checkpoint Visual |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset y formato RLE | 6 imágenes con máscaras + RLE OK |
| **P7** | 9-11 Nov | Baseline U-Net y 1ª Submission | Gráficas de Loss + Submission Kaggle |
| **P8** | 16-18 Nov | Data Augmentation y mejora | Comparativa Baseline vs Augmented |
| **P9** | 23-25 Nov | Inferencia, Threshold y Errores | 5 imágenes normales + 2 casos de error |
| ▶ **P10** | 30 Nov-2 Dic | TTA, Submission Final y Defensa | Mejor Dice Score + Defensa Oral |

> **Regla de Oro:** Según el Art. 7.5 del Reglamento de Evaluación UPCT, la asistencia y validación del Checkpoint en el aula es obligatoria para superar la práctica.


# Práctica 10: TTA, Submission Final y Defensa del Proyecto
## Sesión única (30 Nov Grupo A / 2 Dic Grupo B)

### Objetivos de la sesión:
1. Implementar **Test Time Augmentation (TTA)** para mejorar la robustez de las predicciones en el conjunto de test.
2. Generar la **Submission Final** definitiva para el Leaderboard de Kaggle.
3. Estructurar y ensayar la **Defensa Oral** del proyecto.

### El "Truco" de Kaggle: TTA
En competiciones reales, el modelo no solo predice una vez. Se le pasa la imagen original, su versión volteada, y a veces rotada. Luego se promedian las probabilidades. Esto reduce la varianza y suele subir el Dice Score entre un 1% y un 3%.

> **CHECKPOINT P10:** Mostrar al profesor:
> 1. El código de TTA funcionando.
> 2. La captura de pantalla de vuestra posición final en el Leaderboard de Kaggle.
> 3. El esquema de vuestra presentación oral.

## Bloque 10.1: Test Time Augmentation — el Reverso de Data Augmentation
### TTA es la contrapartida de Data Augmentation, pero en inferencia

En la Práctica 8 entrenasteis con `HorizontalFlip(p=0.5)` para forzar al modelo a ser **invariante** al volteo horizontal: que aprenda que un tumor sigue siendo el mismo tumor, esté en el lado izquierdo o en el derecho de la imagen. TTA explota esa invarianza aprendida, pero en el momento de predecir, no de entrenar:

| | Data Augmentation (P8) | Test Time Augmentation (P10) |
|---|---|---|
| ¿Cuándo se aplica? | Durante el entrenamiento | Durante la inferencia |
| ¿Qué transforma? | Las imágenes de entrenamiento | La imagen de test, varias veces |
| ¿Qué consigue? | Que el modelo *aprenda* a ser invariante | Aprovechar esa invarianza para *promediar* varias opiniones del mismo modelo |
| ¿Cambian los pesos? | Sí, se entrena con ellos | No, el modelo ya está fijo |

> **Idea clave:** si el modelo nunca vio ejemplos volteados horizontalmente durante el entrenamiento, no hay ninguna razón para esperar que su predicción sobre la imagen volteada sea fiable — TTA solo funciona porque la Práctica 8 preparó al modelo para esto.

### Por qué hay que deshacer el flip de la predicción antes de promediar

Cuando volteáis la imagen de entrada y la pasáis por el modelo, la máscara que obtenéis también sale volteada: un tumor que en la imagen original está a la izquierda, en la predicción sobre la imagen volteada aparece "a la derecha" del lienzo. Si promediarais esa predicción directamente con la original (sin devolverla a su orientación), estaríais mezclando el píxel `(x, y)` de una con el píxel `(ancho - x, y)` de la otra — dos posiciones que no representan el mismo punto de la imagen. Por eso el algoritmo pide explícitamente voltear la predicción de vuelta **antes** del promedio: es el mismo principio del Bloque 7.1 (una transformación geométrica debe deshacerse de forma simétrica para que las cosas sigan alineadas).

### Promediar probabilidades, no máscaras ya binarizadas

Fijaos en el orden exacto del algoritmo: `probs_final = (probs_orig + probs_flip) / 2`, y **solo después** se aplica `best_threshold`. Esto no es casualidad — es el mismo principio de "umbralizar una sola vez, al final" que visteis en la Práctica 9. Si primero binarizarais cada predicción por separado y luego promediarais dos máscaras de 0s y 1s, perderíais la información de *cuánta confianza* tenía cada vista: un píxel donde una vista dice 0.9 y la otra 0.6 (tumor claro en ambas) y otro donde una dice 0.51 y la otra 0.49 (frontera dudosa) pueden dar el mismo resultado si binarizáis antes de promediar, pero son situaciones muy distintas si promediáis las probabilidades primero.

### El coste de TTA: tiempo de inferencia, no de entrenamiento

Data Augmentation y TTA tienen el coste computacional en lados opuestos del proceso:

| Técnica | Dónde cuesta más tiempo |
|---------|----------------------------|
| Data Augmentation | Entrenamiento (cada época procesa variaciones distintas) |
| TTA | Inferencia (cada imagen de test se predice 2 veces: original + volteada) |

Con 234 imágenes de test y solo 2 vistas por imagen, el coste extra es asumible en Colab. Si en vez de 1 flip usarais 4 u 8 transformaciones distintas, el tiempo de inferencia crecería proporcionalmente — TTA es una decisión de compromiso entre tiempo disponible y mejora esperada de Dice.

### Por qué horizontal flip y no cualquier otra transformación

En el Bloque 8.1 establecisteis que una rotación de 90° o un flip vertical no son transformaciones clínicamente válidas para ecografía mamaria (cambian la orientación anatómica de forma que nunca ocurriría en una adquisición real) — y por eso el pipeline de augmentation nunca las usó. La misma regla se aplica aquí: TTA solo debería usar transformaciones con las que el modelo **fue entrenado a convivir**. Aplicar TTA con una rotación de 90°, por ejemplo, promediaría una predicción fiable (la original) con una predicción sobre una entrada que el modelo nunca aprendió a interpretar — en el mejor de los casos no ayuda, en el peor, empeora el resultado.

> **Pregunta para pensar:** el enunciado dice que el TTA "suele" subir el Dice entre un 1% y un 3%, no que lo garantice siempre. Según lo anterior, ¿en qué circunstancia concreta esperaríais que el TTA con flip horizontal *no* mejorase, o incluso empeorase el resultado?

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| TTA vs Data Augmentation | DA entrena invarianza; TTA la explota en inferencia, sin tocar los pesos |
| Deshacer el flip | La predicción sobre la imagen volteada también sale volteada; hay que revertirla antes de comparar píxel a píxel |
| Promediar antes de umbralizar | Promediar probabilidades conserva matices de confianza que promediar máscaras binarias pierde |
| Coste | Tiempo extra en inferencia (proporcional al nº de vistas), no en entrenamiento |
| Transformaciones válidas | Solo las que el modelo vio durante el entrenamiento (Bloque 8.1) |

## Tarea 10.1: Implementación de Test Time Augmentation (TTA)
### ¿Qué es el TTA?
Durante el entrenamiento usamos *Data Augmentation* para que el modelo vea más datos.
Durante la inferencia (test), usamos *Test Time Augmentation* para que el modelo "vea" la misma imagen desde diferentes perspectivas y promedie su confianza.

### El algoritmo:
1. Tomar la imagen original de test.
2. Predecir la máscara de probabilidad (`probs_orig`).
3. Voltear la imagen horizontalmente (`flip_img`).
4. Predecir la máscara de la imagen volteada (`probs_flip`).
5. Voltear la predicción de vuelta a la orientación original.
6. **Promediar**: `probs_final = (probs_orig + probs_flip) / 2`.
7. Aplicar el `best_threshold` sobre `probs_final`.

### Instrucciones:
1. Define una función `predict_with_tta(model, img_tensor, device, threshold)`.
2. Dentro, haz la predicción normal.
3. Aplica `torch.flip(img_tensor, [-1])` para voltear la imagen.
4. Haz la segunda predicción y vuelve a voltear el resultado con `torch.flip()`.
5. Promedia ambas salidas y binariza con el umbral.

In [ ]:
# ============================================================
# TAREA 10.1: FUNCIÓN DE INFERENCIA CON TTA
# ============================================================

# ESCRIBE TU CÓDIGO AQUÍ

def predict_with_tta(model, img_tensor, device, threshold=0.5):
    """
    Realiza inferencia usando Test Time Augmentation (Horizontal Flip).
    """
    model.eval()
    with torch.no_grad():
        # 1. Predicción original
        # Tu código aquí...

        # 2. Voltear imagen y predecir
        # img_flipped = torch.flip(img_tensor, [-1])
        # Tu código aquí...

        # 3. Voltear la predicción de vuelta
        # Tu código aquí...

        # 4. Promediar y umbralizar
        # Tu código aquí...

    return mask_pred

print("Función TTA definida")

## Bloque 10.2: Los Últimos Detalles Antes de la Submission Definitiva
### Por qué `dtype=torch.float32` es "CLAVE"

El código marca explícitamente `torch.tensor(img_norm, dtype=torch.float32)`. Esto no es una precaución genérica — sin ello, os encontraríais con un error real y bastante críptico la primera vez que os pase.

`numpy` usa `float64` (doble precisión) por defecto en muchas operaciones aritméticas, mientras que los pesos de vuestro modelo son `float32` (precisión simple) — es el estándar en redes neuronales, porque duplicar la precisión no mejora el resultado y sí duplica la memoria usada. Si construís el tensor de entrada sin fijar el dtype y `numpy` os cuela un `float64` en algún paso intermedio, PyTorch os devolverá algo como `RuntimeError: expected scalar type Float but found Double` en la primera capa convolucional — el modelo no puede multiplicar pesos `float32` por una entrada `float64`.

> **Idea clave:** fijar `dtype=torch.float32` explícitamente al crear el tensor es más seguro que confiar en que todas las operaciones previas (`.astype(np.float32)`, restas, divisiones) mantengan la precisión que esperáis en cada paso.

### Anidar `torch.no_grad()` no es un error

El comentario `# Opcional, ya lo maneja la función, pero buena práctica` señala algo real: `predict_with_tta` ya envuelve su contenido en `torch.no_grad()` (Tarea 10.1). Poner un segundo `with torch.no_grad():` alrededor de la llamada, en el bucle de esta tarea, es redundante pero **no es un error** — los gestores de contexto de PyTorch se pueden anidar sin problema; el segundo `no_grad()` simplemente confirma una condición que ya estaba activa. Es una práctica defensiva razonable: si en el futuro alguien modifica `predict_with_tta` y olvida el `no_grad()` interno, el externo seguiría protegiendo el bucle de inferencia.

### El resize y el segundo threshold, una vez más

En la sección de post-procesado (`mask_resized = cv2.resize(mask_np, ...)` seguido de `mask_binary = (mask_resized > 0.5)`) reconoceréis exactamente el patrón del Bloque 9.3: `predict_with_tta` os devuelve una máscara ya binarizada a resolución `IMG_SIZE`; al redimensionarla al tamaño original con `cv2.resize`, la interpolación reintroduce valores intermedios en los bordes, y por eso hace falta ese segundo `> 0.5` antes de pasarla a `mask_to_rle`. El patrón no cambia por usar TTA — solo cambia que la máscara que redimensionáis ahora viene de promediar dos vistas en vez de una.

### El recorrido completo de una submission

A estas alturas, cada submission que habéis subido a Kaggle representa una capa más sobre la anterior:

```
submission.csv          (P7):  modelo baseline, threshold fijo 0.5
submission_final.csv    (P9):  mejor modelo + best_threshold calibrado
submission_TTA.csv      (P10): mejor modelo + best_threshold + promedio de 2 vistas (TTA)
```

Cada fichero debería, en principio, superar (o al menos igualar) al anterior en el Leaderboard — si no es así, es una señal útil para la Tarea 9.3 (revisar qué está fallando) más que algo que ocultar en la defensa oral.

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| `dtype=torch.float32` | Evita el error de mezclar `float64` (numpy) con los pesos `float32` del modelo |
| `torch.no_grad()` anidado | Seguro y redundante, no un error — refuerza una protección que ya existía |
| Resize + segundo threshold | Mismo patrón del Bloque 9.3: la interpolación reintroduce valores intermedios en los bordes |
| Progresión de submissions | Cada CSV añade una mejora sobre el anterior: baseline → threshold óptimo → TTA |

## Tarea 10.2: Generación de la Submission Final con TTA
Es el momento de la verdad. Vamos a usar la función TTA para generar nuestro último `submission.csv`.

### Instrucciones:
1. Itera sobre las imágenes de `test/images`.
2. Preprocesa la imagen (recuerda el `dtype=torch.float32` para evitar errores de tipo).
3. Llama a `predict_with_tta()` usando vuestro `best_threshold`.
4. Redimensiona la máscara al tamaño original y conviértela a RLE.
5. Guarda el DataFrame como **`submission_TTA.csv`** y súbelo a Kaggle.

> **Pista:** El TTA duplica el tiempo de inferencia, pero en un dataset de 234 imágenes en Colab solo tardará unos minutos extra. ¡Merece la pena!

> **CHECKPOINT P10.1:** Descarga `submission_TTA.csv` y súbelo a Kaggle. Muestra al profesor tu nueva posición en el Leaderboard.

In [ ]:
# ============================================================
# TAREA 10.2: SUBMISSION FINAL CON TTA
# ============================================================
import pandas as pd

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.png')))
results = []

print(f"Generando Submission Final con TTA (Threshold = {best_threshold:.2f})...")
print(f"Total imágenes: {len(test_images)}")

with torch.no_grad(): # Opcional, ya lo maneja la función, pero buena práctica
    for idx, img_path in enumerate(test_images):
        # 1. Cargar y preprocesar
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_norm = img_resized.astype(np.float32) / 255.0

        # Normalización ImageNet
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img_norm - mean) / std

        # CLAVE: dtype=torch.float32
        img_tensor = torch.tensor(img_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        # 2. Predecir con TTA
        # Tu código aquí...

        # 3. Post-procesar y RLE
        # mask_np = mask_pred.squeeze().cpu().numpy()
        # mask_resized = cv2.resize(mask_np, (original_w, original_h))
        # mask_binary = (mask_resized > 0.5).astype(np.uint8)
        # rle = mask_to_rle(mask_binary)
        # results.append({'Id': img_path.name, 'Expected': rle})

        if (idx + 1) % 50 == 0:
            print(f"   Progreso: {idx + 1}/{len(test_images)}")

# Guardar CSV
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_TTA.csv', index=False)
print("¡Submission Final generada! (submission_TTA.csv)")

## Bloque 10.3: Preparar las Preguntas, no Solo las Diapositivas
### Por qué la defensa oral pesa tanto como el código

Un modelo que funciona pero que su autor no puede explicar es, en la práctica profesional, un modelo que no se puede defender ante un comité ético, un radiólogo escéptico o un revisor de un artículo científico. La defensa oral no es un trámite añadido al final del proyecto — es la comprobación de que entendisteis las decisiones que tomasteis, no solo que copiasteis código que funciona.

### La pregunta que rompe una defensa poco preparada: "¿por qué?"

Los criterios de evaluación piden "pensamiento crítico: no solo leer números, sino explicar por qué el modelo falla o acierta". En la práctica, esto se traduce en que el profesor no va a preguntar "¿qué Dice Score obtuvisteis?" (eso ya está en la pantalla) sino "¿por qué ese threshold y no 0.5?", "¿por qué esta imagen falla y esta otra no?". Si solo memorizáis los resultados sin entender las decisiones detrás, esas preguntas os van a pillar en blanco delante del profesor — que es exactamente el momento que cuenta para la nota.

### Batería de preguntas para poneros a prueba antes de la defensa

Repasad estas preguntas por vuestra cuenta antes de la defensa. Cada una se corresponde con un Bloque que ya habéis trabajado — si no sabéis responderla con seguridad, es el bloque que os conviene releer:

| Pregunta | Bloque donde está la respuesta |
|----------|-----------------------------------|
| ¿Por qué normalizáis con media/std de ImageNet y no solo dividiendo por 255? | Bloque 7.1 (Dataset y DataLoader) |
| ¿Por qué la función de pérdida combina Dice + BCE en vez de usar solo una? | Bloque 7.2 (U-Net y Función de Pérdida) |
| ¿Qué diferencia hay entre `model.train()` y `model.eval()`, y por qué importa? | Bloque 7.3 (Bucle de Entrenamiento) |
| ¿Por qué no podéis calcular vuestro Dice real sobre el conjunto de test? | Bloque 7.4 (De Entrenamiento a Submission) |
| ¿Por qué una rotación de 90° no es una augmentation válida aquí? | Bloque 8.1 (Data Augmentation) |
| ¿Por qué hay que "desnormalizar" la imagen para poder visualizarla? | Bloque 8.2 (Verificar Visualmente) |
| ¿Por qué reinicializasteis el modelo en vez de continuar entrenando el baseline? | Bloque 8.3 (Repetir un Experimento) |
| ¿Por qué mirar solo el Val Dice final no basta para concluir que un modelo es mejor? | Bloque 8.4 (Leer una Comparativa) |
| ¿Por qué el threshold se calibra en validación y no en train ni en test? | Bloque 9.1 (El Threshold como Hiperparámetro) |
| ¿Por qué una imagen "normal" nunca puede producir un Falso Negativo? | Bloque 9.2 (Leer los Errores) |
| ¿Por qué hay que redimensionar la máscara con cuidado antes del RLE? | Bloque 9.3 (Cerrar el Círculo) |
| ¿Por qué el TTA con flip horizontal debería funcionar, pero uno con rotación de 90° no? | Bloque 8.1 (Data Augmentation) |

> **Idea clave:** si un compañero de otro grupo os hiciera estas preguntas sin avisar, ¿podríais responder sin mirar el notebook? Esa es la prueba real de si estáis listos para la defensa.

### Cómo responder cuando no sabéis algo

Si el profesor pregunta algo que no recordáis con precisión, la peor respuesta posible es inventar algo que suene bien pero sea incorrecta — un evaluador con experiencia lo detecta enseguida, y penaliza más el intento de "rellenar" que la honestidad. Es mejor decir "no estoy seguro de la razón exacta, pero lo que observamos fue..." y describir lo que sí visteis en vuestros propios resultados. La honestidad técnica es también parte de lo que se evalúa como "claridad técnica".

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Por qué importa la defensa | Un modelo que no podéis explicar no es defendible profesionalmente |
| La pregunta clave | No es "qué resultado obtuvisteis" sino "por qué tomasteis esa decisión" |
| Autoevaluación previa | Repasar la tabla de preguntas por bloque antes de la defensa, no solo las diapositivas |
| Ante una duda real | Honestidad sobre lo observado, mejor que una justificación inventada |  

## Tarea 10.3: Preparación de la Defensa Oral
Habéis hecho el trabajo de ingenieros biomédicos. Ahora toca comunicarlo. La defensa oral es el 50% de la nota de la Práctica 10.

### Estructura recomendada de la presentación (3-5 minutos por persona):

1. **Contexto Clínico (1 min):**
   * ¿Qué es el dataset BUSI? ¿Por qué es importante segmentar tumores mamarios?
   * ¿Qué reto clínico plantean las imágenes "normales"?
2. **EDA y Preprocesamiento (1 min):**
   * Distribución de clases.
   * Explicación del formato RLE y por qué es necesario.
3. **Arquitectura y Entrenamiento (2 min):**
   * ¿Por qué U-Net? (Mencionar Skip Connections).
   * Función de pérdida: ¿Por qué combinamos Dice + BCE?
   * Estrategia de Data Augmentation: ¿Qué funcionó y qué no? (Mostrar gráficas comparativas).
4. **Análisis de Errores y Threshold (1.5 min):**
   * Mostrar la gráfica de optimización del threshold.
   * Mostrar 1 caso de éxito y 1 caso de fallo (Falso Positivo/Negativo) y explicar *clínicamente* por qué ocurrió.
5. **Conclusiones y TTA (0.5 min):**
   * ¿Cuánto mejoró el score final con TTA?
   * ¿Qué haríais si tuvierais un mes más para el proyecto?

### Criterios de Evaluación:
* **Claridad técnica:** Uso correcto de la terminología (Dice, BCE, Overfitting, TTA).
* **Pensamiento crítico:** No solo leer números, sino explicar *por qué* el modelo falla o acierta.
* **Trabajo en equipo:** Ambos miembros del grupo deben hablar.

> **CHECKPOINT P10.2:** Enseña al profesor el esquema de tu presentación antes de salir de clase. ¡Mucha suerte en la defensa!